# 02 — Acquire DGEG trade and sales

Discover and download DGEG petroleum-product trade files plus the long annual sales workbook. Raw files are never modified.


In [ ]:
from pathlib import Path
import re
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.dgeg import (
    canonicalise_trade_long,
    read_sales_workbook,
    read_trade_workbook,
)
from portugal_refining_resilience.sources import discover_download_links, download_file, load_source_manifest, year_from_url
from portugal_refining_resilience.io import sha256_file


In [ ]:
sources = load_source_manifest(ROOT / "config" / "sources.yml")
trade_links = discover_download_links(sources["dgeg_trade"]["landing_page"])
sales_links = discover_download_links(sources["dgeg_sales"]["landing_page"])
print(f"DGEG trade downloads discovered: {len(trade_links)}")
print(f"DGEG sales downloads discovered: {len(sales_links)}")


In [ ]:
# Match the annual product import/export family by name rather than by any year in
# the URL. "dgeg-oip-2000-2024.xlsx" also ends in 2024 but holds crude oil imports by
# country of origin, a different concept, and would otherwise overwrite the 2024
# product workbook at the same target path.
TRADE_WORKBOOK = re.compile(r"/dgeg-oie-(?P<year>\d{4})\.(?P<suffix>xlsx?)$", re.IGNORECASE)

records = []
for url in trade_links:
    match = TRADE_WORKBOOK.search(url.split("?")[0])
    if match is None:
        continue
    year = int(match.group("year"))
    if year < 2000:
        continue
    suffix = "." + match.group("suffix").lower()
    target = PATHS.raw / "dgeg" / "trade" / f"dgeg_trade_{year}{suffix}"
    download_file(url, target)
    records.append({"dataset": "dgeg_trade", "year": year, "path": str(target.relative_to(ROOT)), "url": url, "sha256": sha256_file(target)})

# Prefer the long sales workbook when present.
for url in sales_links:
    if "1970" not in url and "2024" not in url:
        continue
    suffix = ".xlsx" if ".xlsx" in url.lower() else ".xls"
    target = PATHS.raw / "dgeg" / f"dgeg_sales_long{suffix}"
    download_file(url, target)
    records.append({"dataset": "dgeg_sales", "year": None, "path": str(target.relative_to(ROOT)), "url": url, "sha256": sha256_file(target)})
    break

manifest = pd.DataFrame(records)
display(manifest.tail())
persist_dataframe(manifest, PATHS.provenance / "dgeg_download_manifest.csv")


In [ ]:
# Extract each downloaded workbook. DGEG publishes one file per year laying
# partner country against product in tonnes, and every sheet carries its own
# total row, which read_trade_workbook excludes and then uses as a parse check.
trade_frames = []
for record in records:
    if record["dataset"] != "dgeg_trade":
        continue
    workbook = ROOT / record["path"]
    trade_frames.append(read_trade_workbook(workbook, year=int(record["year"])))

if trade_frames:
    dgeg_long = pd.concat(trade_frames, ignore_index=True)
    persist_dataframe(
        dgeg_long,
        PATHS.interim / "dgeg_trade_long.csv",
        key_columns=["year", "product", "flow"],
        metadata={"unit": "toneladas", "source": "DGEG"},
    )
    dgeg_trade = canonicalise_trade_long(dgeg_long)
    persist_dataframe(
        dgeg_trade,
        PATHS.interim / "dgeg_trade_annual_canonical.csv",
        key_columns=["year", "product", "flow"],
        metadata={"unit": "kt", "source": "DGEG"},
    )
    print(f"DGEG trade years: {sorted(dgeg_trade['year'].unique().tolist())}")
    display(dgeg_trade.head())
else:
    print("No DGEG trade workbooks discovered; the trade cross-check remains incomplete.")


In [ ]:
# The long sales workbook runs 1970-2024, so domestic demand has a cross-check
# across the whole study window even though DGEG trade workbooks start at 2019.
# Gasoline is the "Gasolinas" total; diesel needs coloured and marked gasoil added
# to road gasoil. See read_sales_workbook for why.
sales_records = [r for r in records if r["dataset"] == "dgeg_sales"]
if sales_records:
    dgeg_sales = read_sales_workbook(ROOT / sales_records[0]["path"])
    dgeg_sales = dgeg_sales.loc[dgeg_sales["year"].between(2005, 2024)]
    persist_dataframe(
        dgeg_sales,
        PATHS.interim / "dgeg_domestic_sales_annual.csv",
        key_columns=["year", "product"],
        metadata={"unit": "kt", "concept": "domestic market sales"},
    )
    display(dgeg_sales.tail())
else:
    print("No DGEG sales workbook discovered; demand cross-check remains incomplete.")